# 03 — Preprocess & Train

## Goal
Load enriched features from `02_feature_engineering.ipynb`, apply model-specific
preprocessing, run optional Optuna hyperparameter tuning, and save all artifacts
needed by `04_predict_evaluate.ipynb`.

**This notebook is a pure orchestrator — all logic lives in `src/pipeline_preprocess.py`.**

## Inputs
```
outputs/enriched/train_enriched.parquet  — enriched train features (no target)
outputs/enriched/y_train.parquet         — train target aligned by index
outputs/enriched/val_enriched.parquet    — enriched val features (no target)
outputs/enriched/y_val.parquet           — val target aligned by index
outputs/enriched/test_enriched.parquet   — enriched frozen TEST features (no target)
outputs/enriched/y_test.parquet          — frozen TEST target aligned by index
```

## Outputs
```
outputs/preproc/X_train_lgbm.parquet    — label-encoded train features (LightGBM / XGBoost)
outputs/preproc/X_val_lgbm.parquet      — label-encoded val features   (LightGBM / XGBoost)
outputs/preproc/X_test_lgbm.parquet     — label-encoded frozen TEST features
outputs/preproc/X_train_cat.parquet     — raw-categorical train features (CatBoost)
outputs/preproc/X_val_cat.parquet       — raw-categorical val features   (CatBoost)
outputs/preproc/X_test_cat.parquet      — raw-categorical frozen TEST features
outputs/preproc/y_train.parquet         — train target
outputs/preproc/y_val.parquet           — val target
outputs/preproc/y_test.parquet          — frozen TEST target
outputs/preproc/encoding_map.pkl        — fitted label encoders (LightGBM / XGBoost)
outputs/preproc/cat_features.pkl        — categorical column list (CatBoost)
outputs/best_params_lgbm.json           — best Optuna params for LightGBM
outputs/best_params_xgb.json            — best Optuna params for XGBoost
```


In [ ]:
# Present Project's structure
import os
import sys

# Notebook lives in v2/ & v3/— go one level up to reach project root
PROJECT_ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))

for subdir in ['', 'v2', 'v3', 'src']:  
    p = os.path.join(PROJECT_ROOT_PATH, subdir)
    if p not in sys.path:
        sys.path.insert(0, p)


from project_utils import print_project_structure
print_project_structure()

PROJECT STRUCTURE
Root: C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection
├── ARCHIVE
│   ├── OLD_outputs
│   │   ├── enriched
│   │   │   ├── test_enriched.parquet
│   │   │   ├── train_enriched.parquet
│   │   │   └── y_train.parquet
│   │   ├── models
│   │   │   ├── cat_cols.pkl
│   │   │   ├── model_cat.cbm
│   │   │   ├── model_lgbm.pkl
│   │   │   └── model_xgb.pkl
│   │   ├── preproc
│   │   │   ├── encoding_map.pkl
│   │   │   ├── new_feature_cols.pkl
│   │   │   ├── X_train_lgbm.parquet
│   │   │   ├── X_train_raw.parquet
│   │   │   ├── X_val_lgbm.parquet
│   │   │   ├── X_val_raw.parquet
│   │   │   ├── y_train.parquet
│   │   │   └── y_val.parquet
│   │   ├── submissions
│   │   │   └── submission_lgbm_v2.csv
│   │   ├── best_params_lgbm.json
│   │   └── best_params_xgb.json
│   ├── OLD_src
│   │   ├── config.py  ← shared — paths, constants, seeds
│   │   ├── data_loader.py  ← shared — load/merge/save raw and processed data
│   │   ├── eda.py
│   │   ├── evaluate_ml.p

## Step 0 — Setup

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Notebook lives in v2/ — go one level up to reach project root
PROJECT_ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
for subdir in ['', 'src', 'v0']:
    p = os.path.join(PROJECT_ROOT_PATH, subdir)
    if p not in sys.path:
        sys.path.insert(0, p)

from config import (
    PROJECT_ROOT, NON_FEATURE_COLS,
    ENRICHED_DIR, PREPROC_DIR, OUTPUTS_DIR
)
from pipeline_preprocess import (
    load_enriched,
    preprocess_and_save,
    load_preprocessed,
    preprocess_catboost_and_save,
    load_catboost_preprocessed,
    run_optuna_lgbm,
)
from tune_optuna_with_early_stop import tune_xgb

print(f'Project root : {PROJECT_ROOT}')
print(f'Enriched dir : {ENRICHED_DIR}')
print(f'Preproc dir  : {PREPROC_DIR}')


Project root : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection
Enriched dir : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\enriched
Preproc dir  : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc


## Step 1 — Run control flags

### Preprocessing flags
- `PREPROC_READY_LGBM_XGB = True` — skip LightGBM / XGBoost preprocessing, load saved files

> **Note:** LightGBM and XGBoost share identical preprocessing (label encoding + fill NaN).
> They use the same files: `X_train_lgbm.parquet` / `X_val_lgbm.parquet`.


In [2]:
# ── Preprocessing flags ───────────────────────────────────────────────────
# LightGBM and XGBoost share identical preprocessing (label encoding + fill NaN).
# They use the same saved files: X_train_lgbm.parquet / X_val_lgbm.parquet.
PREPROC_READY_LGBM_XGB = False   # True → skip preprocessing, load saved files

# ── Optuna flags ───────────────────────────────────────────────────────────
RUN_OPTUNA_LGBM = False   # True → tune LightGBM, save best_params_lgbm.json
RUN_OPTUNA_XGB  = False   # True → tune XGBoost,  save best_params_xgb.json

# ── Quality / latency profile ──────────────────────────────────────────────
# "min"      — ~0.7h    | ~0.004 AUC loss vs high
# "med"      — ~3h      | ~0.001 AUC loss vs high
# "med_high" — ~8-10h   | ~0.0005 AUC loss vs high
# "high"     — ~18h     | reference (no loss)
QUALITY = "med_high"

print(f'PREPROC_READY_LGBM_XGB : {PREPROC_READY_LGBM_XGB}')
print(f'RUN_OPTUNA_LGBM        : {RUN_OPTUNA_LGBM}')
print(f'RUN_OPTUNA_XGB         : {RUN_OPTUNA_XGB}')
print(f'QUALITY                : {QUALITY}')


PREPROC_READY_LGBM_XGB : False
RUN_OPTUNA_LGBM        : False
RUN_OPTUNA_XGB         : False
QUALITY                : med_high


## Step 2 — Load enriched data

In [3]:
if not PREPROC_READY_LGBM_XGB:
    train_enriched, y_train, val_enriched, y_val, test_enriched, y_test = load_enriched(ENRICHED_DIR)
else:
    print('PREPROC_READY_LGBM_XGB=True — skipping data load.')


STEP 1 — Load enriched data (train / val / frozen TEST)
   train_enriched : (354324, 462)  | fraud rate: 3.3833%
   val_enriched   : (118108, 462)    | fraud rate: 3.9041%
   test_enriched  : (118108, 462)   | fraud rate: 3.4409%  (frozen TEST)
   Index alignment: OK ✓


## Step 4 — Model-specific preprocessing

Each model has its own preprocessing pipeline and its own `PREPROC_READY` flag.
Set the flag to `True` to skip preprocessing and load saved files directly.

In [4]:
import pandas as pd

# ── Step 4: LightGBM / XGBoost preprocessing ──────────────────────────────
# LightGBM and XGBoost share identical preprocessing: label encoding + fill NaN.
# Output: X_train_lgbm.parquet, X_val_lgbm.parquet, X_test_lgbm.parquet, encoding_map.pkl
if not PREPROC_READY_LGBM_XGB:
    X_train_lgbm, X_val_lgbm, X_test_lgbm, encoding_map = preprocess_and_save(
        X_train=train_enriched, X_val=val_enriched, X_test=test_enriched,
        y_train=y_train, y_val=y_val, y_test=y_test,
        preproc_dir =PREPROC_DIR,
        cols_to_drop=NON_FEATURE_COLS,
    )
else:
    X_train_lgbm, X_val_lgbm, X_test_lgbm, encoding_map, y_train, y_val, y_test = load_preprocessed(PREPROC_DIR)


STEP 2 — Preprocess and save (train / val / frozen TEST)
PREPROCESSING — FIT (LightGBM / XGBoost)
>> Encoding categorical columns (fit on train)...
   Shape before: (354324, 462)
   Found 32 categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5']...
   Encoded 32 columns
   Shape after: (354324, 462)

>> Filling missing values with -1...
   NaN before: 62,083,199 values in 342 columns
   NaN after:  0

>> Dropping non-feature columns...
   Shape before: (354324, 462)
   Dropped: ['TransactionID', 'TransactionDT']
   Not found (skipped): ['isFraud']
   Shape after: (354324, 460)

   Final shape: (354324, 460)
   Dtypes: {dtype('float32'): np.int64(422), dtype('int32'): np.int64(34), dtype('int64'): np.int64(2), dtype('int16'): np.int64(1), dtype('int8'): np.int64(1)}
PREPROCESSING — TRANSFORM (LightGBM / XGBoost)
>> Encoding categorical columns (transform)...
   Shape before: (118108, 462)
   Transformed 32 columns
   Unseen v

## Step 5 — Optuna hyperparameter tuning

Runs Bayesian TPE search for each model independently.
Each model runs only if its `RUN_OPTUNA_*` flag is `True`.
Best params are saved to JSON and loaded automatically by `train_*.py`.

**Quality profiles** (set `QUALITY` in Step 1):
| Profile | Trials | Data fraction | Time    | AUC loss vs high |
|---------|--------|---------------|---------|------------------|
| `min`   | 30     | 30%           | ~0.7h   | ~0.004           |
| `med`   | 50     | 50%           | ~3h     | ~0.001           |
| `high`  | 100    | 100%          | ~18h    | 0 (reference)    |

### Step 5.1 — Optuna: LightGBM

In [5]:
if RUN_OPTUNA_LGBM:
    run_optuna_lgbm(
        X_train_lgbm, y_train,
        X_val_lgbm,   y_val,
        outputs_dir=OUTPUTS_DIR,
        quality=QUALITY,
    )
else:
    params_path = OUTPUTS_DIR / 'best_params_lgbm.json'
    if params_path.exists():
        print(f'RUN_OPTUNA_LGBM=False — using saved params:\n  {params_path}')
    else:
        print('RUN_OPTUNA_LGBM=False and no saved params found.')
        print('train_lightgbm.py will use DEFAULT_PARAMS.')
        print(f'Expected: {params_path}')

RUN_OPTUNA_LGBM=False — using saved params:
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\best_params_lgbm.json


### Step 5.2 — Optuna: XGBoost

In [6]:
if RUN_OPTUNA_XGB:
    tune_xgb(
        X_train_lgbm, y_train,
        X_val_lgbm,   y_val,
        quality=QUALITY,
        save_path=str(OUTPUTS_DIR / 'best_params_xgb.json'),
    )
else:
    params_path = OUTPUTS_DIR / 'best_params_xgb.json'
    if params_path.exists():
        print(f'RUN_OPTUNA_XGB=False — using saved params:\n  {params_path}')
    else:
        print('RUN_OPTUNA_XGB=False and no saved params found.')
        print('train_xgboost.py will use DEFAULT_PARAMS.')
        print(f'Expected: {params_path}')

RUN_OPTUNA_XGB=False — using saved params:
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\best_params_xgb.json


## Step 6 — Summary

In [7]:
from pipeline_preprocess import print_preprocessing_summary

print_preprocessing_summary(
    X_train_lgbm, X_val_lgbm, X_test_lgbm,
    y_train, y_val, y_test,
    encoding_map,
    preproc_dir=PREPROC_DIR,
    outputs_dir=OUTPUTS_DIR,
)

PREPROCESSING SUMMARY
  X_train_lgbm : (354324, 460)  | fraud rate: 3.3833%
  X_val_lgbm   : (118108, 460)    | fraud rate: 3.9041%
  X_test_lgbm  : (118108, 460)   | fraud rate: 3.4409%  (frozen TEST)
  encoding_map : 32 label encoders fitted on train

  Dtypes in X_train_lgbm:
    float32     : 422 columns
    int32       : 34 columns
    int64       : 2 columns
    int16       : 1 columns
    int8        : 1 columns

  NaN in X_train_lgbm : 0 (expected: 0)
  NaN in X_val_lgbm   : 0   (expected: 0)
  NaN in X_test_lgbm  : 0  (expected: 0)
  NaN check: OK ✓

  Saved files:
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\X_train_lgbm.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\X_val_lgbm.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\X_test_lgbm.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\y_train.parquet
    ✓  C:\Users\USER\Desktop\NO